# Weather API project: Medallion architecture (Bronze layer) using OpenMeteo API
### by Matias Bertuzzi
### Data Engineer | SunnyData

#1. Configuration

In [0]:
import requests
from datetime import datetime, timezone
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType, DoubleType)

In [0]:
dbutils.widgets.text("environment", "dev", "Environment")
environment = dbutils.widgets.get("environment")
catalog = f"mbertuzzi_{environment}"
dbutils.widgets.text("schema", "weatherapi", "Schema")
dbutils.widgets.text("bronze_table", "bronze_weather_raw", "Bronze table name")
dbutils.widgets.text("location_name", "Buenos Aires", "Location name")
dbutils.widgets.text("latitude", "-34.6037", "Latitude")
dbutils.widgets.text("longitude", "-58.3816", "Longitude")

In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
bronze_table = dbutils.widgets.get("bronze_table")
location_name = dbutils.widgets.get("location_name")
latitude = dbutils.widgets.get("latitude")
longitude = dbutils.widgets.get("longitude")

full_table_name = f"{catalog}.{schema}.{bronze_table}"
print(f"Target table: {full_table_name}")

In [0]:
if spark.sql(f"SHOW CATALOGS LIKE '{catalog}'").count() > 0:
    print("Catalog already exists")
else:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

# 2. Extract

In [0]:
API_URL = "https://api.open-meteo.com/v1/forecast"
 
params = {
    "latitude": latitude,
    "longitude": longitude,
    "current_weather": "true",
    "hourly": "temperature_2m,relative_humidity_2m,precipitation,weathercode",
    "timezone": "auto",
}
 
response = requests.get(API_URL, params=params, timeout=30)
response.raise_for_status()
 
raw_json = response.text  # keep the exact payload, unmodified
ingestion_ts = datetime.now(timezone.utc)
 
print(f"Pulled {len(raw_json)} bytes from {response.url}")

In [0]:
schema_bronze = StructType([
    StructField("raw_response", StringType(), False),
    StructField("source_url", StringType(), False),
    StructField("location_name", StringType(), False),
    StructField("latitude", DoubleType(), False),
    StructField("longitude", DoubleType(), False),
    StructField("ingestion_timestamp", TimestampType(), False),
])
 
row = Row(
    raw_response=raw_json,
    source_url=response.url,
    location_name=location_name,
    latitude=float(latitude),
    longitude=float(longitude),
    ingestion_timestamp=ingestion_ts,
)

In [0]:
df = spark.createDataFrame([row], schema=schema_bronze)

# 3. Write

In [0]:
(
    df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(full_table_name)
)
 
print(f"Appended 1 row to {full_table_name} at {ingestion_ts.isoformat()}")

In [0]:
# display(
#         spark.sql(f"SELECT * FROM {full_table_name} ORDER BY ingestion_timestamp DESC LIMIT 5")
#         )